contracts renewed before 30 days of ced

In [ ]:
import pandas as pd
from functions import search_files, clean_ft
pd.set_option('display.max_columns', None)


In [ ]:
bein_src = pd.read_csv(search_files("BEINDATANEWRPT")[1], dtype='str')


In [ ]:
bein_src.loc[bein_src['Customer Number']=='19194985']

In [ ]:
filename = search_files('NEWCNEFIN')[0]
ft_src =clean_ft(filename)

In [41]:
ft_src.loc[ft_src['Subscriber Nr']=='19192028']

,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,User Name,User Fullname,Payment Ref No,Payment Batch No,Batch Approval No,Default Entity Type,Collecting Entity,Pay Mode,Jv Type,Book Number,Smartcard,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,PAYMENT_FLAG,DS_PAYMENT_FLAG
212445,19192028,Invoice,INV_9284929,16-09-2026 00:00:00,03:28:07 PM,Posted,05-10-2026 00:00:00,04-10-2027 00:00:00,,6087,KTAYEF,Kholoud Tayef Ahmed,,,,CNE Head Office,,,,,10732761373,05/10/2026 - 04/10/2027,12M,Subscription Invoice,PREMIUM 09.24,4709810,beIN,CNE Subscriber,New Cairo 4,,,
684088,19192028,JV,CNEJVR_541301,07-04-2026 00:00:00,08:57:56 AM,Posted,,,,6087,SFEKY,Shereen abd allah El-Feky,,,,CNE Head Office,,,Credit,,10732761373,,,,,,beIN,CNE Subscriber,New Cairo 4,,,
792200,19192028,Invoice,INV_8822302,06-04-2026 00:00:00,02:54:35 PM,Posted,06-04-2026 00:00:00,05-10-2026 00:00:00,,5677,cnenewc2user2,EDD NEW CAIRO 2 user2,,,,CNE Dealer,,,,,10732761373,06/04/2026 - 05/10/2026,6M,Subscription Invoice,ULTIMATE 6 Months Contract,4121069,beIN,CNE Subscriber,New Cairo 4,,,
792268,19192028,JV,CNEJVN_605444,06-04-2026 00:00:00,02:54:36 PM,Posted,,,,5677,cnenewc2user2,EDD NEW CAIRO 2 user2,,,,CNE Dealer,,,Credit,,10732761373,,,,,,beIN,CNE Subscriber,New Cairo 4,,,


In [30]:
bein = bein_src.copy()
ft = ft_src.copy()
ft['Created Date'] = pd.to_datetime(ft['Created Date'], dayfirst=True)
bein['Start Date'] = pd.to_datetime( bein['Start Date'], dayfirst=True,errors='coerce')
bein['End Date'] = pd.to_datetime( bein['End Date'], dayfirst=True,errors='coerce')
bein['Next Billing Date'] = pd.to_datetime( bein['Next Billing Date'], dayfirst=True,errors='coerce')

main_pck_filter = (
            bein["Plan"].str.contains(
                "prem",
                case=False,
                na=False
            )
            |
            bein["Plan"].str.contains(
                "ulti",
                case=False,
                na=False
            )
            |
            bein["Plan"].str.contains(
                "toget",
                case=False,
                na=False
            )
        )

offer_pck_filter = ~bein["Plan"].str.contains('btl',case=False)

bein = bein.loc[main_pck_filter & offer_pck_filter & (bein['Customer Type'].isin(dth_types)) & (bein['Status']=='Active')]


In [31]:
ft.loc[ft['Subscriber Nr']=='19194985']

,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,User Name,User Fullname,Payment Ref No,Payment Batch No,Batch Approval No,Default Entity Type,Collecting Entity,Pay Mode,Jv Type,Book Number,Smartcard,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,PAYMENT_FLAG,DS_PAYMENT_FLAG
108399,19194985,Invoice,INV_9327434,2026-09-30,05:30:46 PM,Posted,06-10-2026 00:00:00,05-04-2027 00:00:00,,3045,cnemnfyuser2,EDD MENOFIYA user2,,,,CNE Dealer,,,,,10732917397,06/10/2026 - 05/04/2027,6M,Subscription Invoice,Premium 6 Months,4772643,beIN,CNE Subscriber,El Minofia,,,
217452,19194985,JV,CNEJVN_697230,2026-09-30,05:30:46 PM,Posted,,,,3045,cnemnfyuser2,EDD MENOFIYA user2,,,,CNE Dealer,,,Credit,,10732917397,,,,,,beIN,CNE Subscriber,El Minofia,,,
233830,19194985,Invoice,INV_8720813,2026-02-21,12:33:14 AM,Posted,21-02-2026 00:00:00,20-11-2026 00:00:00,,3990,cnemnfyuser1,EDD MENOFIYA user1,,,,CNE Dealer,,,,,10732917397,21/02/2026 - 20/11/2026,9M,Subscription Invoice,AddON FWC 2026,4068537,beIN,CNE Subscriber,El Minofia,,,
233985,19194985,JV,CNEJVN_594249,2026-02-21,12:33:16 AM,Posted,,,,3990,cnemnfyuser1,EDD MENOFIYA user1,,,,CNE Dealer,,,Credit,,10732917397,,,,,,beIN,CNE Subscriber,El Minofia,,,


In [32]:
grouped = bein.groupby('Customer Number').agg(count = ('Contract Number','count')).reset_index()

grouped = grouped.loc[grouped['count']>1]

In [33]:
bein = bein.loc[bein['Customer Number'].isin(grouped['Customer Number'])]

In [34]:
bein = bein.sort_values(['Customer Number', 'End Date'], ascending=[True,True]).drop_duplicates(subset=['Customer Number'])


In [35]:
ft.loc[ft['Subscriber Nr']=='19194985']

,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,User Name,User Fullname,Payment Ref No,Payment Batch No,Batch Approval No,Default Entity Type,Collecting Entity,Pay Mode,Jv Type,Book Number,Smartcard,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,PAYMENT_FLAG,DS_PAYMENT_FLAG
108399,19194985,Invoice,INV_9327434,2026-09-30,05:30:46 PM,Posted,06-10-2026 00:00:00,05-04-2027 00:00:00,,3045,cnemnfyuser2,EDD MENOFIYA user2,,,,CNE Dealer,,,,,10732917397,06/10/2026 - 05/04/2027,6M,Subscription Invoice,Premium 6 Months,4772643,beIN,CNE Subscriber,El Minofia,,,
217452,19194985,JV,CNEJVN_697230,2026-09-30,05:30:46 PM,Posted,,,,3045,cnemnfyuser2,EDD MENOFIYA user2,,,,CNE Dealer,,,Credit,,10732917397,,,,,,beIN,CNE Subscriber,El Minofia,,,
233830,19194985,Invoice,INV_8720813,2026-02-21,12:33:14 AM,Posted,21-02-2026 00:00:00,20-11-2026 00:00:00,,3990,cnemnfyuser1,EDD MENOFIYA user1,,,,CNE Dealer,,,,,10732917397,21/02/2026 - 20/11/2026,9M,Subscription Invoice,AddON FWC 2026,4068537,beIN,CNE Subscriber,El Minofia,,,
233985,19194985,JV,CNEJVN_594249,2026-02-21,12:33:16 AM,Posted,,,,3990,cnemnfyuser1,EDD MENOFIYA user1,,,,CNE Dealer,,,Credit,,10732917397,,,,,,beIN,CNE Subscriber,El Minofia,,,


In [36]:
ft = ft.loc[ft['Subscriber Nr'].isin(grouped['Customer Number'])]
ft = ft.loc[(ft['Doc Type'].isin(['Payment','JV'])) & (ft['Doc Status'] == 'Posted')]
ft = ft.sort_values(['Subscriber Nr','Created Date'],ascending=[True,False])
ft = ft.drop_duplicates(subset=['Subscriber Nr'],keep='first')
ft.loc[ft['Subscriber Nr']=='19194985']

,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,User Name,User Fullname,Payment Ref No,Payment Batch No,Batch Approval No,Default Entity Type,Collecting Entity,Pay Mode,Jv Type,Book Number,Smartcard,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,PAYMENT_FLAG,DS_PAYMENT_FLAG
217452,19194985,JV,CNEJVN_697230,2026-09-30,05:30:46 PM,Posted,,,,3045,cnemnfyuser2,EDD MENOFIYA user2,,,,CNE Dealer,,,Credit,,10732917397,,,,,,beIN,CNE Subscriber,El Minofia,,,


In [37]:
ft.loc[ft['Subscriber Nr']=='19194985']

,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,User Name,User Fullname,Payment Ref No,Payment Batch No,Batch Approval No,Default Entity Type,Collecting Entity,Pay Mode,Jv Type,Book Number,Smartcard,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,PAYMENT_FLAG,DS_PAYMENT_FLAG
217452,19194985,JV,CNEJVN_697230,2026-09-30,05:30:46 PM,Posted,,,,3045,cnemnfyuser2,EDD MENOFIYA user2,,,,CNE Dealer,,,Credit,,10732917397,,,,,,beIN,CNE Subscriber,El Minofia,,,


In [38]:
bein = bein.merge(right=ft[['Subscriber Nr','Created Date','User Name','Default Entity Type', 'Collecting Entity']], left_on='Customer Number', right_on='Subscriber Nr', how='inner')

In [39]:
bein['difference'] = (  bein['End Date'] - bein['Created Date'] ).dt.days
bein = bein.loc[bein['difference']>30]

bein['User Name'] = bein['User Name'].str.upper()


users_allocation = pd.read_csv('users_allocation.csv', dtype='str')

bein = bein.merge(right=users_allocation[['User Name',  'DEPT']], on='User Name',how='left')


bein[['Customer Number', 'Customer Type', 'Entity', 'Contract Number',
       'Start Date', 'End Date', 'Plan', 'Status', 'Decoder',
        'Smart Card', 
       'Next Billing Date',  'Created Date', 'User Name',
       'Default Entity Type', 'Collecting Entity', 'difference','DEPT']].to_csv('more than 30 days.csv', index=False)

In [40]:
# ft_grouped = ft.groupby(['Smartcard']).agg(last_payment = ('Created Date','max')).reset_index()